# 05 · Appendix D, Table D6 – cross-model agreement in the time-fraction elicitation

Fills the per-model rows of Table D6 (and documents the Gemini overshoot mentioned in Section 3.1.1). It only needs `data/elicited_time_fractions.csv` and `data/final_task_register.csv`, both handed off by `code/data_preparation/` (notebooks 01 and 03). Independent of notebooks 01–04.

Output: `outputs/4_appendix_D/tableD6_time_fraction_agreement.csv`

In [1]:
from pathlib import Path
import pandas as pd
import r2r_common as rc

TF_CSV = rc.DATA_DIR / "elicited_time_fractions.csv"                       # columns: task_id, occupation_title, claude_hours, gpt_hours, gemini_hours, ...
REGISTER_CSV = rc.DATA_DIR / "final_task_register.csv"                      # phase 1 register, handed off by data_preparation/03

tf = pd.read_csv(TF_CSV, dtype={"task_id": str})
model_cols = ["claude_hours", "gpt_hours", "gemini_hours"]
print("Rows:", len(tf), "| rows with all 3 models:", tf[model_cols].dropna().shape[0])

Rows: 212 | rows with all 3 models: 212


In [2]:
try:
    import krippendorff
    alpha = krippendorff.alpha(reliability_data=tf[model_cols].to_numpy(dtype=float).T, level_of_measurement="ordinal")
    print(f"Krippendorff's alpha (ordinal), should reproduce 0.681: {alpha:.3f}")
except ImportError:
    alpha = float("nan")
    print("pip install krippendorff to recompute alpha (reported value: 0.681)")


def agreement_rows(df, scope):
    others_max = df[["claude_hours", "gpt_hours"]].max(axis=1)
    occ_totals = df.groupby("occupation_title")[model_cols].sum().mean()
    row = {"scope": scope, "rows": len(df)}
    for c in model_cols:
        m = c.replace("_hours", "")
        row[f"mean_hours_{m}"] = df[c].mean()
        row[f"median_hours_{m}"] = df[c].median()
        row[f"mean_total_hours_per_occupation_{m}"] = occ_totals[c]
    row["share_gemini_strictly_highest_pct"] = 100 * (df["gemini_hours"] > others_max).mean()
    row["share_gemini_highest_incl_ties_pct"] = 100 * (df["gemini_hours"] >= others_max).mean()
    return row


reg = pd.read_csv(REGISTER_CSV, dtype={"task_id": str})
in_scope = set(reg.loc[reg["final_label"] != "OUT_OF_SCOPE", "task_id"].str.strip())
tableD6 = pd.DataFrame([agreement_rows(tf, "all 212 task-occupation rows"),
                        agreement_rows(tf[tf["task_id"].isin(in_scope)], "91 in-scope tasks")]).round(2)
tableD6["krippendorff_alpha_ordinal_all_rows"] = round(alpha, 3)
rc.save_csv(tableD6, "appD", "tableD6_time_fraction_agreement.csv")
tableD6.T

Krippendorff's alpha (ordinal), should reproduce 0.681: 0.681
[write] outputs/4_appendix_D/tableD6_time_fraction_agreement.csv


,0,1
scope,all 212 task-occupation rows,91 in-scope tasks
rows,212,91
mean_hours_claude,2.73,3.38
median_hours_claude,2.0,3.0
mean_total_hours_per_occupation_claude,57.87,30.8
mean_hours_gpt,2.13,2.66
median_hours_gpt,1.5,2.5
mean_total_hours_per_occupation_gpt,45.23,24.24
mean_hours_gemini,3.76,4.46
median_hours_gemini,3.0,4.0
